# The Heritage Short — Classical Ethiopian Music Generation (Meta MusicGen / AudioCraft)

Generates a culturally authentic classical Ethiopian score matching the `storyboard.json` contract:
- **Cue 1 (Struggle):** Solo Krar plucked lyre in D minor Tizita, muted Kebero hand drum rhythm, 74 BPM.
- **Cue 2 (Wisdom):** Washint bamboo flute, resonant Begena box lyre in G major Tizita, 80 BPM.
- **Duration:** Exactly 66.0s (matching the film's total run duration).

**Run on:** Kaggle Free T4 GPU (takes ~45 seconds).

In [ ]:
# Step 1: Install AudioCraft & modern PyAV dependencies
# 1. System FFmpeg libs
!apt-get -qq install -y ffmpeg pkg-config libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev libavfilter-dev

# 2. Prebuilt current PyAV wheel (avoids compiling old pinned wheel)
!pip install -q av "numpy<2"

# 3. AudioCraft without its pinned dependencies, then its runtime deps
!pip install -q --no-deps git+https://github.com/facebookresearch/audiocraft.git
!pip install -q einops encodec flashy hydra-core hydra_colorlog julius num2words sentencepiece librosa soundfile "transformers>=4.40" demucs omegaconf huggingface_hub

import torch
import torchaudio
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Load Meta MusicGen Medium model on GPU
print("Loading Meta MusicGen Medium (1.5B parameters)...")
model = MusicGen.get_pretrained('facebook/musicgen-medium')

# Set generation duration to 66 seconds (covers 65.5s narrative film)
model.set_generation_params(duration=66)
print("Model loaded successfully on GPU!")

In [ ]:
# Step 3: Storyboard-conditioned acoustic prompt
descriptions = [
    "Traditional Ethiopian acoustic folk instrumental, solo krar plucked bowl lyre in D minor Tizita, "
    "muted kebero hand drum rhythm, evolving into warm washint bamboo flute and resonant begena box lyre in G major Tizita, "
    "ancestral highland atmosphere, contemplative, no vocals, pure acoustic instrumental"
]

print("Generating 66-second classical Ethiopian score...")
wav = model.generate(descriptions)

# Step 4: Export broadcast-ready score_final.wav
audio_write('score_final', wav[0].cpu(), model.sample_rate, strategy="loudness", loudness_compressor=True)
print("SUCCESS! Generated: score_final.wav")

In [ ]:
# Step 5: Listen inline to the generated score
import IPython.display as ipd
ipd.Audio('score_final.wav')